# 004 · Clocks and Measuring Time

> **Standard Library › Dates & Times** · Python 3.14

The `time` module exposes several different clocks. Picking the right one matters: the wall clock can jump (NTP sync, the user changing the time), so it is the wrong tool for measuring how long something took — the same reason .NET has `Stopwatch` alongside `DateTime.UtcNow`.

**In this notebook you will learn:**

- the four clocks — `time()`, `monotonic()`, `perf_counter()` and `process_time()` — and when to use each
- the integer `*_ns` variants
- `time.sleep()` and its .NET equivalents
- how to build a reusable timing context manager (your own `Stopwatch`)
- a first look at `timeit` for micro-benchmarks

## .NET mental model

| Need | .NET | Python |
|---|---|---|
| Current wall-clock time | `DateTime.UtcNow` / `DateTimeOffset.UtcNow` | `time.time()` (or `datetime.now(timezone.utc)`) |
| Unix time in ms | `DateTimeOffset.UtcNow.ToUnixTimeMilliseconds()` | `time.time_ns() // 1_000_000` |
| Coarse monotonic ticks | `Environment.TickCount64` | `time.monotonic()` |
| High-resolution timing | `Stopwatch` / `Stopwatch.GetTimestamp()` | `time.perf_counter()` |
| CPU time used by the process | `Process.GetCurrentProcess().TotalProcessorTime` | `time.process_time()` |
| Pause the thread | `Thread.Sleep(ms)` | `time.sleep(seconds)` |
| Micro-benchmarks | BenchmarkDotNet (NuGet) | `timeit` |

Python has no `Stopwatch` class, but one is about ten lines of code — we build it below.

## `time.time()` — the wall clock

`time.time()` returns seconds since the Unix epoch (1970-01-01 UTC) as a `float`. It is what you want for *timestamps*, and what you must **not** use for *durations*: the system clock can be adjusted forwards or backwards at any moment.

```csharp
// C#
long nowMs = DateTimeOffset.UtcNow.ToUnixTimeMilliseconds();
```

In [ ]:
import time
from datetime import datetime, timezone

now = time.time()
print(type(now).__name__)                                     # float — seconds since the epoch
print(now > 1_700_000_000)                                    # True (any date after Nov 2023)
print(datetime.fromtimestamp(now, tz=timezone.utc).tzinfo)    # convert to an aware datetime when needed

## `time.monotonic()` — never goes backwards

A **monotonic** clock only moves forward and is not affected by changes to the system time. Its absolute value is meaningless (it often counts from boot); only the *difference* between two readings matters. It is the analogue of `Environment.TickCount64`, but returned in fractional seconds.

Use it for timeouts and deadlines:

In [ ]:
deadline = time.monotonic() + 0.2          # "give up 200 ms from now"
attempts = 0
while time.monotonic() < deadline:
    attempts += 1
    time.sleep(0.05)                        # pretend to poll something

print(f"stopped after ~{attempts} attempts")   # ≈ 4 (timing-dependent)

## `time.perf_counter()` — the stopwatch

`perf_counter()` is monotonic too, with the **highest available resolution**. It is the default choice for measuring how long code takes — Python's `Stopwatch.GetTimestamp()`.

```csharp
// C#
var sw = Stopwatch.StartNew();
DoWork();
Console.WriteLine($"took {sw.Elapsed.TotalMilliseconds:F1} ms");

// or, allocation-free (.NET 7+)
long start = Stopwatch.GetTimestamp();
DoWork();
TimeSpan elapsed = Stopwatch.GetElapsedTime(start);
```

In [ ]:
start = time.perf_counter()
total = sum(i * i for i in range(200_000))
elapsed = time.perf_counter() - start

print(f"computed {total} in {elapsed * 1000:.1f} ms")   # duration varies by machine
print(elapsed > 0)                                       # True

## `time.process_time()` — CPU time

`process_time()` counts CPU time consumed by the current process (user + system), and **does not advance while sleeping** or waiting for I/O. Compare it with wall time to see whether code is CPU-bound or waiting:

In [ ]:
wall_start, cpu_start = time.perf_counter(), time.process_time()
time.sleep(0.2)                                         # waiting: no CPU used
wall, cpu = time.perf_counter() - wall_start, time.process_time() - cpu_start
print(f"sleep: wall ≈ {wall:.1f}s, cpu is a tiny fraction of that: {cpu < wall / 2}")

wall_start, cpu_start = time.perf_counter(), time.process_time()
sum(i * i for i in range(1_000_000))                    # busy: burns CPU
wall, cpu = time.perf_counter() - wall_start, time.process_time() - cpu_start
print(f"busy:  cpu used: {cpu > 0}")

`time.thread_time()` is the same idea for the current thread only.

## Inspecting the clocks

`time.get_clock_info(name)` tells you whether a clock is monotonic, whether it can be adjusted, and its resolution on this platform (the resolution and the underlying OS call vary by machine):

In [ ]:
for name in ["time", "monotonic", "perf_counter", "process_time"]:
    info = time.get_clock_info(name)
    print(f"{name:13} monotonic={info.monotonic!s:5}  adjustable={info.adjustable}")

Choosing a clock:

| You want… | Use |
|---|---|
| a timestamp to store or display | `time.time()` / `datetime.now(timezone.utc)` |
| a timeout or deadline | `time.monotonic()` |
| to measure how long code took | `time.perf_counter()` |
| to measure CPU effort, ignoring waits | `time.process_time()` |

⚠️ **Gotcha:** never measure durations with `time.time()` or `datetime.now()`. A clock adjustment during the measurement gives you negative or wildly wrong results.

## The `*_ns` variants

Every clock has an integer nanosecond twin: `time_ns()`, `monotonic_ns()`, `perf_counter_ns()`, `process_time_ns()`. A 64-bit `float` holding ~1.7 billion seconds only has about a quarter of a microsecond of precision left, so the `_ns` versions avoid rounding errors — handy for exact timestamps and for very short measurements.

In [ ]:
ns = time.time_ns()
print(type(ns).__name__)                   # int
print(ns // 1_000_000 > 1_700_000_000_000) # Unix milliseconds, like ToUnixTimeMilliseconds()

start = time.perf_counter_ns()
_ = [0] * 1_000
duration_ns = time.perf_counter_ns() - start
print(f"{duration_ns:,} ns", duration_ns >= 0)   # value varies

## `time.sleep()`

`time.sleep(seconds)` blocks the current thread. It takes **seconds as a float**, not milliseconds like `Thread.Sleep(int)` — `time.sleep(500)` sleeps for over 8 minutes!

```csharp
// C#
Thread.Sleep(250);                         // milliseconds
Thread.Sleep(TimeSpan.FromMilliseconds(250));
await Task.Delay(250);                     // async version
```

In [ ]:
start = time.perf_counter()
time.sleep(0.25)                           # 250 ms
print(f"slept ≈ {time.perf_counter() - start:.2f}s")   # ≈ 0.25 (never less, may be a little more)

⚠️ **Gotcha:** inside `async` code use `await asyncio.sleep(...)` — the equivalent of `await Task.Delay(...)`. `time.sleep` would block the whole event loop *(see Syntax › Async › 001 · Async and Await)*.

## A reusable `Stopwatch`: a timing context manager

A context manager is the natural Python shape for "time this block" — `__enter__` starts the clock, `__exit__` stops it, even if the block raises *(see Syntax › Error Handling › 003 · Context Managers)*.

In [ ]:
from datetime import timedelta

class Stopwatch:
    """Times a with-block using perf_counter (like System.Diagnostics.Stopwatch)."""

    def __init__(self, label: str = "block", *, report: bool = True):
        self.label, self.report = label, report
        self.elapsed = 0.0

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info):
        self.elapsed = time.perf_counter() - self._start
        if self.report:
            print(f"{self.label}: {self.elapsed * 1000:.0f} ms")
        return False                      # don't swallow exceptions

    @property
    def elapsed_timedelta(self) -> timedelta:
        return timedelta(seconds=self.elapsed)

In [ ]:
with Stopwatch("sleepy work") as sw:      # prints ≈ "sleepy work: 100 ms"
    time.sleep(0.1)

print(sw.elapsed >= 0.1)                  # True — sleep never returns early
print(type(sw.elapsed_timedelta).__name__)

Because `__exit__` always runs, the timing is recorded even when the block fails:

In [ ]:
try:
    with Stopwatch("failing work", report=False) as sw:
        time.sleep(0.05)
        raise RuntimeError("boom")
except RuntimeError as e:
    print(f"{type(e).__name__}: {e} — still timed: {sw.elapsed >= 0.05}")

**Tip:** the same thing written with `contextlib.contextmanager` is shorter, but the class version lets you read `elapsed` after the block, which is usually what you want.

In [ ]:
from contextlib import contextmanager

@contextmanager
def timed(label: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {(time.perf_counter() - start) * 1000:.0f} ms")

with timed("quick nap"):                  # prints ≈ "quick nap: 50 ms"
    time.sleep(0.05)

## A taste of `timeit`

One `perf_counter` measurement is noisy. For comparing small snippets, `timeit` runs the code many times and minimises overhead — a lightweight, built-in cousin of BenchmarkDotNet. It is covered properly in *(see Standard Library › Testing & Debugging › 005 · Timing and Profiling)*.

In [ ]:
import timeit

loops = 1_000
genexpr = timeit.timeit("''.join(str(i) for i in range(100))", number=loops)
listcomp = timeit.timeit("''.join([str(i) for i in range(100)])", number=loops)

# same result, two spellings — values vary by machine
print(f"join(generator):     {genexpr / loops * 1e6:.1f} µs per loop")
print(f"join(list):          {listcomp / loops * 1e6:.1f} µs per loop")

In Jupyter you can also use the `%timeit` magic, e.g. `%timeit sorted(data)` — it chooses the number of loops automatically. It is IPython-only, so it will not work in a plain `.py` file.

## The legacy `time` struct API

The `time` module also has C-style functions (`time.localtime()`, `time.gmtime()`, `time.strftime()`, `time.struct_time`). They predate `datetime`; prefer `datetime` for anything beyond quick scripts.

In [ ]:
print(time.gmtime(0))                                  # struct_time for the Unix epoch
print(time.strftime("%Y-%m-%d", time.gmtime(0)))       # 1970-01-01

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Unix timestamp | `DateTimeOffset.UtcNow.ToUnixTimeSeconds()` | `time.time()` |
| Unix timestamp (exact, ns) | `DateTime.UtcNow.Ticks` (100 ns) | `time.time_ns()` |
| Monotonic ticks | `Environment.TickCount64` | `time.monotonic()` |
| Measure elapsed time | `Stopwatch.StartNew()` / `.Elapsed` | `start = time.perf_counter()` … `time.perf_counter() - start` |
| CPU time | `Process.TotalProcessorTime` | `time.process_time()` |
| Sleep | `Thread.Sleep(250)` | `time.sleep(0.25)` |
| Async delay | `await Task.Delay(250)` | `await asyncio.sleep(0.25)` |
| Time a block | `using` + custom disposable | `with Stopwatch(...)` context manager |
| Micro-benchmark | BenchmarkDotNet | `timeit` / `%timeit` |

## Exercises

1. Write a function `retry_until(predicate, timeout: float, interval: float = 0.01) -> bool` that calls `predicate()` repeatedly until it returns `True` or `timeout` seconds have passed (use the right clock!). Test it with a predicate that becomes true after 50 ms.
2. Turn the timing idea into a **decorator** `@timed_call` that prints how long each call of the decorated function took, and returns the function's result unchanged.
3. Use `time.process_time()` and `time.perf_counter()` to show that `time.sleep(0.1)` uses almost no CPU, by printing the ratio `cpu / wall`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import functools
import time

# 1 — monotonic is the right clock for deadlines
def retry_until(predicate, timeout: float, interval: float = 0.01) -> bool:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if predicate():
            return True
        time.sleep(interval)
    return False

ready_at = time.monotonic() + 0.05
print(retry_until(lambda: time.monotonic() >= ready_at, timeout=1))    # True
print(retry_until(lambda: False, timeout=0.05))                        # False

# 2 — wraps keeps the name and docstring; finally reports even on errors
def timed_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"{func.__name__} took {(time.perf_counter() - start) * 1000:.1f} ms")
    return wrapper

@timed_call
def slow_add(a, b):
    time.sleep(0.02)
    return a + b

print(slow_add(2, 3))     # prints the timing, then 5

# 3 — sleeping costs wall time but (almost) no CPU time
wall0, cpu0 = time.perf_counter(), time.process_time()
time.sleep(0.1)
wall, cpu = time.perf_counter() - wall0, time.process_time() - cpu0
print(f"cpu / wall = {cpu / wall:.3f}")   # close to 0
```

</details>